In [1]:
import numpy as np
import time
import sys

from io import UnsupportedOperation
import pickle as pkl


# Add shared location for auxillary functions
sys.path.insert(1, './../../AuxillaryFunctions')

# import personal Functions
from EvaluationMethodFuncs import KyritEval_PtVsWindow
from EvaluationMethodFuncs import DongEval_PtVsPt
from EvaluationMethodFuncs import TimePoint2Window
from EvaluationMethodFuncs import Window2TimePoint
from EvaluationMethodFuncs import PrintStats_SingleLine


import csv



print("Finished importing libraries.")


Finished importing libraries.


In [2]:
### Demo of reading a file ###

# My_File = "./FiveFold_CTCResults_v1/Clem_Intake/fold0/Clemson_p410_c2_grm_std_uni.csv"
My_File = "./FiveFold_CTCResults_v1/THO_Intake/fold0/OREBA-DIS_1116_1_grm_std_uni.csv"



def ReadPredictionsFileForDetections(My_File):
	with open(My_File, newline='') as csvfile:
		spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
		cnt=0
		Detections=[]
		for row in spamreader:
			cnt+=1
			if (row[4]!='1'): # Fifth Column is either 1 for null action or 2 for intake event
				# print(f"{cnt/8}")
				Detections.append(cnt/8)
			#end if
		#end for row
	#end with open(.csv)
	
	return Detections
# end of ReadPredictionsFileForDetections()

In [3]:
### Grab list of all files and their corresponding ground truth ###

ResultsFilesAllFolds = [
	[
		'OREBA-DIS_1005_1_grm_std_uni.csv',  'OREBA-DIS_1011_1_grm_std_uni.csv',  	'OREBA-DIS_1016_1_grm_std_uni.csv',  
		'OREBA-DIS_1021_1_grm_std_uni.csv',  'OREBA-DIS_1026_1_grm_std_uni.csv',  	'OREBA-DIS_1031_1_grm_std_uni.csv',  
		'OREBA-DIS_1037_1_grm_std_uni.csv',  'OREBA-DIS_1044_1_grm_std_uni.csv',  	'OREBA-DIS_1050_1_grm_std_uni.csv',  
		'OREBA-DIS_1055_1_grm_std_uni.csv',	'OREBA-DIS_1061_1_grm_std_uni.csv',	'OREBA-DIS_1072_1_grm_std_uni.csv',
		'OREBA-DIS_1079_1_grm_std_uni.csv','OREBA-DIS_1084_1_grm_std_uni.csv',	'OREBA-DIS_1089_1_grm_std_uni.csv',
		'OREBA-DIS_1094_1_grm_std_uni.csv','OREBA-DIS_1099_1_grm_std_uni.csv',	'OREBA-DIS_1104_1_grm_std_uni.csv',
		'OREBA-DIS_1110_1_grm_std_uni.csv',	'OREBA-DIS_1116_1_grm_std_uni.csv'
	],
	
	[
		'OREBA-DIS_1001_1_grm_std_uni.csv',	'OREBA-DIS_1006_1_grm_std_uni.csv',	'OREBA-DIS_1012_1_grm_std_uni.csv',
		'OREBA-DIS_1017_1_grm_std_uni.csv',	'OREBA-DIS_1022_1_grm_std_uni.csv',	'OREBA-DIS_1027_1_grm_std_uni.csv',
		'OREBA-DIS_1032_1_grm_std_uni.csv',	'OREBA-DIS_1039_1_grm_std_uni.csv',	'OREBA-DIS_1045_1_grm_std_uni.csv',
		'OREBA-DIS_1051_1_grm_std_uni.csv',	'OREBA-DIS_1056_1_grm_std_uni.csv',	'OREBA-DIS_1063_1_grm_std_uni.csv',
		'OREBA-DIS_1073_1_grm_std_uni.csv',	'OREBA-DIS_1080_1_grm_std_uni.csv',	'OREBA-DIS_1085_1_grm_std_uni.csv',
		'OREBA-DIS_1090_1_grm_std_uni.csv',	'OREBA-DIS_1095_1_grm_std_uni.csv',	'OREBA-DIS_1100_1_grm_std_uni.csv',
		'OREBA-DIS_1105_1_grm_std_uni.csv',	'OREBA-DIS_1111_1_grm_std_uni.csv'
	],
	
	[
		'OREBA-DIS_1002_1_grm_std_uni.csv',	'OREBA-DIS_1007_1_grm_std_uni.csv',	'OREBA-DIS_1013_1_grm_std_uni.csv',
		'OREBA-DIS_1018_1_grm_std_uni.csv',	'OREBA-DIS_1023_1_grm_std_uni.csv',	'OREBA-DIS_1028_1_grm_std_uni.csv',
		'OREBA-DIS_1033_1_grm_std_uni.csv',	'OREBA-DIS_1040_1_grm_std_uni.csv',	'OREBA-DIS_1046_1_grm_std_uni.csv',
		'OREBA-DIS_1052_1_grm_std_uni.csv',	'OREBA-DIS_1057_1_grm_std_uni.csv',	'OREBA-DIS_1064_1_grm_std_uni.csv',
		'OREBA-DIS_1075_1_grm_std_uni.csv',	'OREBA-DIS_1081_1_grm_std_uni.csv',	'OREBA-DIS_1086_1_grm_std_uni.csv',
		'OREBA-DIS_1091_1_grm_std_uni.csv',	'OREBA-DIS_1096_1_grm_std_uni.csv',	'OREBA-DIS_1101_1_grm_std_uni.csv',
		'OREBA-DIS_1107_1_grm_std_uni.csv',	'OREBA-DIS_1112_1_grm_std_uni.csv'
	],
	
	[
		'OREBA-DIS_1003_1_grm_std_uni.csv',	'OREBA-DIS_1008_1_grm_std_uni.csv',	'OREBA-DIS_1014_1_grm_std_uni.csv',
		'OREBA-DIS_1019_1_grm_std_uni.csv',	'OREBA-DIS_1024_1_grm_std_uni.csv',	'OREBA-DIS_1029_1_grm_std_uni.csv',
		'OREBA-DIS_1035_1_grm_std_uni.csv',	'OREBA-DIS_1041_1_grm_std_uni.csv',	'OREBA-DIS_1047_1_grm_std_uni.csv',
		'OREBA-DIS_1053_1_grm_std_uni.csv',	'OREBA-DIS_1059_1_grm_std_uni.csv',	'OREBA-DIS_1067_1_grm_std_uni.csv',
		'OREBA-DIS_1076_1_grm_std_uni.csv',	'OREBA-DIS_1082_1_grm_std_uni.csv',	'OREBA-DIS_1087_1_grm_std_uni.csv',
		'OREBA-DIS_1092_1_grm_std_uni.csv',	'OREBA-DIS_1097_1_grm_std_uni.csv',	'OREBA-DIS_1102_1_grm_std_uni.csv',
		'OREBA-DIS_1108_1_grm_std_uni.csv',	'OREBA-DIS_1113_1_grm_std_uni.csv'
	],
	
	[
		'OREBA-DIS_1004_1_grm_std_uni.csv',	'OREBA-DIS_1010_1_grm_std_uni.csv',	'OREBA-DIS_1015_1_grm_std_uni.csv',
		'OREBA-DIS_1020_1_grm_std_uni.csv',	'OREBA-DIS_1025_1_grm_std_uni.csv',	'OREBA-DIS_1030_1_grm_std_uni.csv',
		'OREBA-DIS_1036_1_grm_std_uni.csv',	'OREBA-DIS_1043_1_grm_std_uni.csv',	'OREBA-DIS_1048_1_grm_std_uni.csv',
		'OREBA-DIS_1054_1_grm_std_uni.csv',	'OREBA-DIS_1060_1_grm_std_uni.csv',	'OREBA-DIS_1068_1_grm_std_uni.csv',
		'OREBA-DIS_1077_1_grm_std_uni.csv',	'OREBA-DIS_1083_1_grm_std_uni.csv',	'OREBA-DIS_1088_1_grm_std_uni.csv',
		'OREBA-DIS_1093_1_grm_std_uni.csv',	'OREBA-DIS_1098_1_grm_std_uni.csv',	'OREBA-DIS_1103_1_grm_std_uni.csv',
		'OREBA-DIS_1109_1_grm_std_uni.csv',	'OREBA-DIS_1115_1_grm_std_uni.csv'
	]
]

print("Finished Defining Fold Filepaths.")

Finished Defining Fold Filepaths.


In [4]:
##### Grab GT Values from the Pickle Database

DATABASE_FILEPATH = "./../../Pickle_Databases/OREBA.pkl"

with open(DATABASE_FILEPATH,'rb') as fh:
	dataset = pkl.load(fh)
#OREBA_Cucumber={'UniqueID','proc_data','handedness','bites_gt'}


# # Grab EvalContent to Have
# STRIDE_sec=5
# CUT_sec=5
# FOLD_INDEX =0
# FOLDS_TOTAL=1
# RR_FLAG = 1
# RESAMPLE_FLAG_FREQ=16

# compiled_eval_data = GenerateEvalData_OREBA(
# 			int(round(CUT_sec*DataFreq)), int(round(STRIDE_sec*DataFreq)),
# 			DATABASE_FILEPATH,
# 			FOLD_INDEX, FOLDS_TOTAL, FOLD_SPLIT=RR_FLAG,
# 			RESAMPLE_FLAG=RESAMPLE_FLAG_FREQ # Frequency of Data Collection in [Hz]
# 			#, RESAMPLE_FLAG=0, SMOOTHING=0 # optional flags not currently used for Paper Experiment
# 			)



print("Finished importing OREBA Pickle.")


Finished importing OREBA Pickle.


In [16]:
# Float value used to determine how many sec the detection can be away from teh window
WIN_TOLERANCE = 8
EvalSelect = 2
# 1 = Dong Eval, 2 = Kyritsis Window Tol


# Global Results from all folds (each element is a list from each fold)
All_All_TP = []
All_All_FP = []
All_All_FN = []


for currFold in range(5):

	# Fold number to evaluate. Valid range is [0,4]
	FOLD_SELECT = currFold

	# Directory containing all results for the specified fold
	# PredDirPath = './FiveFold_CTCResults_v1/THO_Intake/fold{}/'.format(FOLD_SELECT)
	PredDirPath = '/home/jpjolly/CTC_Rouast_Take2/ctc-intake-detection-master/Preds_THO_v4/Fold{}/'.format(FOLD_SELECT)

	# Select Current Fold Results Filenames
	ResultsFiles = ResultsFilesAllFolds[FOLD_SELECT]
	# Create Sanity Check of All Participant and Meal IDs from filenames 
	#     to check against pickle Unique IDs later
	mealID_Check = []
	for filename in ResultsFiles:
		filenameComponents = filename.split("_") 
		mealID_Check.append(filenameComponents[-5]+ "_" + filenameComponents[-4])
	# end of for filename


	# Define blank sets for results
	All_TP=[]
	All_FP=[]
	All_FN=[]

	for idx in range(0, len(ResultsFiles)):
		pickle_idx = (idx * 5)+((-1+FOLD_SELECT)%5) # get current file index in pickle dataset
		currFilename = PredDirPath + ResultsFiles[idx]

		currDets = np.array(ReadPredictionsFileForDetections(currFilename))
		if EvalSelect==1: # Using Dong Eval
			currGT = Window2TimePoint(dataset['bites_gt'][pickle_idx])
		elif EvalSelect==2: # Using Kyritsis Eval with tolerance
			currGT = dataset['bites_gt'][pickle_idx]
		#end EvalSelect switch

		currPickleID = dataset['UniqueID'][pickle_idx]
		currFileID = mealID_Check[idx]
		if currPickleID != currFileID:
			print("ERROR: Mismatched FileID ({}) and MealID ({}).".format(currFileID, currPickleID))
		# end of Pickle ID Sanity check


		if EvalSelect==1: # Using Dong Eval
			[TP, FP, FN, _] = DongEval_PtVsPt(currDets, currGT)
		elif EvalSelect==2: # Using Kyritsis Eval with tolerance
			[TP, FP, FN, FP1, FP2, _] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE=WIN_TOLERANCE)
			# [TP, FP, FN, FP1, FP2, Key] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE = 0.0, BEFORE_TOL = 0.0, AFTER_TOL = 0.0):
		# end of EvalSelect switch 
		All_TP.append(TP)
		All_FP.append(FP)
		All_FN.append(FN)

		# PrintStats_SingleLine(TP, FP, FN)
	# end of for idx

	All_All_TP.append(All_TP)
	All_All_FP.append(All_FP)
	All_All_FN.append(All_FN)

# end of for fold

print("Complete Evaluating Meals.")

Complete Evaluating Meals.


In [17]:
#### Caclulate total performance on data

Final_TP = 0
Final_FP = 0
Final_FN = 0

for i in range(5):
	Final_TP += sum(All_All_TP[i]) 
	Final_FP += sum(All_All_FP[i]) 
	Final_FN += sum(All_All_FN[i])
# end of for fold loop 


PrintStats_SingleLine(Final_TP, Final_FP, Final_FN)


89.470 86.827 92.279 4159   348    631   
